In [1]:
from spark_session import spark

26/06/03 18:59:51 WARN Utils: Your hostname, oscar resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/03 18:59:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/tushar/lake-forge/lakehouse/lakehouse_v2/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/tushar/.ivy2/cache
The jars for the packages stored in: /home/tushar/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-915ab219-92ab-40ca-9d1f-83bcaf8c43d0;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.106.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found software.amazon.awssdk#bundle;2.31.67 in central
	found software.amaz

In [2]:
# tables
tables = [
    "customer",
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]
data_dir_path = "/home/tushar/lake-forge/data_generator/data/data_sf_10"

In [3]:
total_data_size_bytes = 0
for i, table in enumerate(tables, start=1):
    print(f"#{i}/{len(tables)} {table} table")

    src_count = spark.read.parquet(f"{data_dir_path}/{table}.parquet").count()
    print(f"Records count in source parquet file: {src_count}")

    iceberg_count = spark.sql(f"select count(*) from nessie.tpch.{table}").collect()[0][0]
    print(f"Records count in iceberg table: {iceberg_count}")

    size_bytes = spark.sql(f"""
        SELECT SUM(file_size_in_bytes)
        FROM nessie.tpch.{table}.files
    """).collect()[0][0]
    total_data_size_bytes += size_bytes
    print(f"Total data size in iceberg: {size_bytes/1024/1024:.2f} MB ({size_bytes} bytes)")
    print("-"*50)

print(f"Total data size in iceberg for all tables: {total_data_size_bytes/1024/1024:.2f} MB ({total_data_size_bytes} bytes)")


#1/8 customer table


Records count in source parquet file: 1500000


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Records count in iceberg table: 1500000
Total data size in iceberg: 76.68 MB (80403977 bytes)
--------------------------------------------------
#2/8 lineitem table
Records count in source parquet file: 59986052
Records count in iceberg table: 59986052
Total data size in iceberg: 1502.10 MB (1575065430 bytes)
--------------------------------------------------
#3/8 nation table
Records count in source parquet file: 25
Records count in iceberg table: 25
Total data size in iceberg: 0.00 MB (2605 bytes)
--------------------------------------------------
#4/8 orders table
Records count in source parquet file: 15000000
Records count in iceberg table: 15000000
Total data size in iceberg: 363.10 MB (380737035 bytes)
--------------------------------------------------
#5/8 part table
Records count in source parquet file: 2000000
Records count in iceberg table: 2000000
Total data size in iceberg: 40.26 MB (42219545 bytes)
--------------------------------------------------
#6/8 partsupp table
Reco

In [4]:
# snapshots
print("Snapshots")
spark.sql("SELECT * FROM nessie.tpch.customer.snapshots").show()

# files
print("Files")
spark.sql("select * from nessie.tpch.customer.files").show()

# history
print("History")
spark.sql("select * from nessie.tpch.customer.history").show()

Snapshots
+--------------------+-------------------+---------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+---------+---------+--------------------+--------------------+
|2026-06-03 18:51:...|4143244842678646932|     NULL|   append|s3://warehouse/tp...|{spark.app.id -> ...|
+--------------------+-------------------+---------+---------+--------------------+--------------------+

Files
+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+--------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|        column_sizes|    

## Schema Evolution and Time Travel

In [5]:
spark.sql("select * from nessie.tpch.customer").show(5)

+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|        1|Customer#000000001|   IVhzIApeRb ot,c,E|         15|25-989-741-2988|   711.56|    BUILDING|to the even, regu...|
|        2|Customer#000000002|XSTf4,NCwDVaWNe6t...|         13|23-768-687-3665|   121.65|  AUTOMOBILE|l accounts. blith...|
|        3|Customer#000000003|        MG9kdTD2WBHm|          1|11-719-748-3364|  7498.12|  AUTOMOBILE| deposits eat sly...|
|        4|Customer#000000004|         XxVSJsLAGtn|          4|14-128-190-5944|  2866.83|   MACHINERY| requests. final,...|
|        5|Customer#000000005|KvpyuHCplrB84WgAi...|          3|13-750-942-6364|   794.47|   HOUSEHOLD|n accounts will h...|
+-------

In [6]:
spark.sql("alter table nessie.tpch.customer add column ingestion_ts timestamp")

DataFrame[]

In [7]:
from pyspark.sql.functions import current_timestamp

spark.table("nessie.tpch.customer").withColumn("ingestion_ts", current_timestamp()).writeTo("nessie.tpch.customer").overwritePartitions()

In [8]:
spark.sql("select * from nessie.tpch.customer").show(5)

+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|        ingestion_ts|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|        1|Customer#000000001|   IVhzIApeRb ot,c,E|         15|25-989-741-2988|   711.56|    BUILDING|to the even, regu...|2026-06-03 18:54:...|
|        2|Customer#000000002|XSTf4,NCwDVaWNe6t...|         13|23-768-687-3665|   121.65|  AUTOMOBILE|l accounts. blith...|2026-06-03 18:54:...|
|        3|Customer#000000003|        MG9kdTD2WBHm|          1|11-719-748-3364|  7498.12|  AUTOMOBILE| deposits eat sly...|2026-06-03 18:54:...|
|        4|Customer#000000004|         XxVSJsLAGtn|          4|14-128-190-5944|  2866.83|   MACHINERY| requests. final,...|2026-06

In [9]:
spark.sql("select * from nessie.tpch.customer.snapshots").show(5, truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list   

In [10]:
snapshot_ids = spark.sql("select snapshot_id from nessie.tpch.customer.snapshots").collect()
snapshot_id_0, snapshot_id_1 = snapshot_ids[0][0], snapshot_ids[1][0]

# customer table right now
print(f"Customer table right now (snapshot_id: {snapshot_id_1}):")
spark.sql("select * from nessie.tpch.customer").show(5)

# customer table before adding timestamp column and overwriting partitions
print(f"Customer table before adding timestamp column (snapshot_id: {snapshot_id_0}):")
spark.sql(f"select * from nessie.tpch.customer version as of {snapshot_id_0}").show(5)


Customer table right now (snapshot_id: 8451267406450910101):
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|        ingestion_ts|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|        1|Customer#000000001|   IVhzIApeRb ot,c,E|         15|25-989-741-2988|   711.56|    BUILDING|to the even, regu...|2026-06-03 18:54:...|
|        2|Customer#000000002|XSTf4,NCwDVaWNe6t...|         13|23-768-687-3665|   121.65|  AUTOMOBILE|l accounts. blith...|2026-06-03 18:54:...|
|        3|Customer#000000003|        MG9kdTD2WBHm|          1|11-719-748-3364|  7498.12|  AUTOMOBILE| deposits eat sly...|2026-06-03 18:54:...|
|        4|Customer#000000004|         XxVSJsLAGtn|          4|14-128

## Branching in Nessie

In [9]:
# list branches
spark.conf.get("spark.sql.catalog.nessie.uri")

# print("Branches:")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

spark.sql("CREATE BRANCH if not exists experiment IN nessie").show()

+-------+----------+----------------------------------------------------------------+
|refType|name      |hash                                                            |
+-------+----------+----------------------------------------------------------------+
|Branch |experiment|584788d123cfba3d4274c30cf9884ad3d6c8145ec4ee6b1a39b5b81e426d6f48|
|Branch |main      |584788d123cfba3d4274c30cf9884ad3d6c8145ec4ee6b1a39b5b81e426d6f48|
+-------+----------+----------------------------------------------------------------+

+-------+----------+--------------------+
|refType|      name|                hash|
+-------+----------+--------------------+
| Branch|experiment|584788d123cfba3d4...|
+-------+----------+--------------------+



In [31]:
spark.sql("USE REFERENCE experiment IN nessie").show()
spark.sql("create table if not exists nessie.tpch.users (id int, name string)").show()
spark.sql("insert into nessie.tpch.users values (1, 'Alice')")
spark.sql("select * from nessie.tpch.users").show()
spark.sql("show tables in nessie.tpch").show()

+-------+----------+--------------------+
|refType|      name|                hash|
+-------+----------+--------------------+
| Branch|experiment|d2f6c1163bb62a31e...|
+-------+----------+--------------------+

++
||
++
++



26/06/03 19:10:07 WARN S3FileIO: Unclosed S3FileIO instance created by:
	org.apache.iceberg.aws.s3.S3FileIO.initialize(S3FileIO.java:507)
	org.apache.iceberg.CatalogUtil.loadFileIO(CatalogUtil.java:423)
	org.apache.iceberg.CatalogUtil.loadFileIO(CatalogUtil.java:370)
	org.apache.iceberg.nessie.NessieCatalog.initialize(NessieCatalog.java:132)
	org.apache.iceberg.CatalogUtil.loadCatalog(CatalogUtil.java:295)
	org.apache.iceberg.CatalogUtil.buildIcebergCatalog(CatalogUtil.java:352)
	org.apache.iceberg.spark.SparkCatalog.buildIcebergCatalog(SparkCatalog.java:154)
	org.apache.iceberg.spark.SparkCatalog.initialize(SparkCatalog.java:774)
	org.apache.spark.sql.execution.datasources.v2.NessieCatalogBridge.setCurrentRefForSpark(NessieCatalogBridge.java:105)
	org.apache.spark.sql.execution.datasources.v2.UseReferenceExec.runInternal(UseReferenceExec.scala:44)
	org.apache.spark.sql.execution.datasources.v2.NessieExec.run(NessieExec.scala:34)
	org.apache.spark.sql.execution.datasources.v2.V2Command

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  1|Alice|
|  1|Alice|
|  1|Alice|
|  1|Alice|
|  1|Alice|
+---+-----+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     tpch| customer|      false|
|     tpch| lineitem|      false|
|     tpch|   nation|      false|
|     tpch|   orders|      false|
|     tpch|     part|      false|
|     tpch| partsupp|      false|
|     tpch|   region|      false|
|     tpch| supplier|      false|
|     tpch|    users|      false|
+---------+---------+-----------+



In [29]:
spark.sql("USE REFERENCE main IN nessie").show()

+-------+----+--------------------+
|refType|name|                hash|
+-------+----+--------------------+
| Branch|main|27960ef4b71a15ab5...|
+-------+----+--------------------+



In [30]:
spark.sql("show tables in nessie.tpch").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     tpch| customer|      false|
|     tpch| lineitem|      false|
|     tpch|   nation|      false|
|     tpch|   orders|      false|
|     tpch|     part|      false|
|     tpch| partsupp|      false|
|     tpch|   region|      false|
|     tpch| supplier|      false|
|     tpch|    users|      false|
+---------+---------+-----------+

